# M1~M4 랜덤포레스트 모델링

로지스틱/LightGBM M1~M4와 **동일한 train/test 분할**(random_state=42) 재사용.

**로드맵 대응** (기법은 전 모델 랜덤포레스트로 통일 — 로지스틱 버전은 별도 파일):

| 모델 | X | 역할 |
|---|---|---|
| M1 | A_financial | 베이스라인 — 금융정보만의 기준점 AUC |
| M2 | B_nonfinancial_existing | 참고치 (RQ 직접 검증용 아님) |
| M3 | A+B | RQ1 핵심 — 비금융 추가 시 예측력 상승 확인 |
| M4 | A+B+C | RQ2 — 신규 비금융(C) 추가 시 예측력 상승 확인 |

**RQ 판정 기준**
- RQ1 핵심: M1 vs M3 (비금융 B 추가 효과)
- RQ2 핵심: M3 vs M4 (신규비금융 C만의 순수 추가 효과 — B는 이미 M3에 포함돼있어 통제됨)
- M1 vs M4는 B+C 효과가 섞여있어 RQ1·RQ2 어느 쪽 판정 근거로도 쓰지 않음 (참고용 총괄 수치만)

①AUC 상승 ②DeLong 유의 ③실질적 상승폭 ④비금융 변수 중요도 비중을 자동으로 체크. ④는 로지스틱 계수 p-value 기준이라 랜덤포레스트에는 해당 없어 변수중요도 비중으로 대체.

8번 단계에서 저장되는 feature_importance CSV에는 각 변수가 A/B/C 어느 군인지 표시하는 "군" 컬럼 포함.

3번 단계에서 `warm_start=True`로 나무 개수별 AUC 추이를 먼저 확인하고, 그래프에서 AUC가 평평해지는 지점을 `N_ESTIMATORS_FINAL`에 반영한 뒤 4번부터 이어서 실행.

## 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 한글 폰트 설정 (Colab 환경)
!apt-get -qq install fonts-nanum > /dev/null 2>&1
import matplotlib.font_manager as fm

font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fm.fontManager.addfont(font_path)
plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

BASE_DIR = "/content/drive/MyDrive/BOOSTMAP/데이터/완료"  # 본인 환경에 맞게 수정

A_PATH = os.path.join(BASE_DIR, "model_A_financial_final.csv")
B_PATH = os.path.join(BASE_DIR, "model_B_nonfinancial_final.csv")
C_PATH = os.path.join(BASE_DIR, "model_C_nonfinancial_new_final.csv")

TARGET = "TARGET"
RANDOM_STATE = 42
TEST_SIZE = 0.2

## 데이터 로드 + 트리모델용 변수 정리

In [ ]:
df_a = pd.read_csv(A_PATH)
df_b = pd.read_csv(B_PATH)
df_c = pd.read_csv(C_PATH)

# 로그변환 컬럼은 원본과 IV/그룹기여도가 동일하고, 랜덤포레스트는 왜도가 큰
# 원본 변수도 다수의 트리로 충분히 걸러낼 수 있어 제외 (원본만 사용)
LOG_COLS = ["AMT_CREDIT_LOG", "AMT_ANNUITY_LOG", "AMT_GOODS_PRICE_LOG", "AMT_INCOME_TOTAL_LOG"]
df_a = df_a.drop(columns=LOG_COLS)

# AGE_BAND는 object(문자열) 타입이라 이번 수치형 기반 모델링에서는 제외
df_b = df_b.drop(columns=["AGE_BAND"])

A_VARS = [c for c in df_a.columns if c not in ("SK_ID_CURR", TARGET)]
B_VARS = [c for c in df_b.columns if c not in ("SK_ID_CURR", TARGET)]
C_VARS = [c for c in df_c.columns if c not in ("SK_ID_CURR", TARGET)]
print(f"A_금융 {len(A_VARS)}개 / B_기존비금융 {len(B_VARS)}개 / C_신규비금융 {len(C_VARS)}개 / "
      f"A+B {len(A_VARS) + len(B_VARS)}개 / A+B+C {len(A_VARS) + len(B_VARS) + len(C_VARS)}개 "
      f"(트리모델이라 더미트랩 처리 없이 전부 사용)")

y = df_a[TARGET].values
print(f"{len(y):,}행, 부도율 {y.mean():.2%}")

## train/test 분할 (로지스틱·LightGBM과 동일 분할 재사용)

In [ ]:
idx_train, idx_test = train_test_split(
    df_a.index, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
y_train, y_test = y[idx_train], y[idx_test]


def make_X(df, cols):
    return df.loc[:, cols].replace([np.inf, -np.inf], np.nan).fillna(0)


X_A = make_X(df_a, A_VARS)
X_B = make_X(df_b, B_VARS)
X_C = make_X(df_c, C_VARS)
X_AB = pd.concat([X_A, X_B], axis=1)
X_ABC = pd.concat([X_A, X_B, X_C], axis=1)

## (진단) n_estimators별 AUC 추이 확인 — 최적 지점 탐색

In [ ]:
# warm_start=True: 이전에 만든 나무는 그대로 두고 늘어난 만큼만 추가로 학습
# → 매번 처음부터 다시 학습하지 않아도 되므로, 여러 지점을 한 번에 확인 가능
# M4(A+B+C, RQ2 보강용 모델) 기준으로 확인 — 여기서 확인한 지점을 4번의
# N_ESTIMATORS_FINAL 값으로 반영하면 됨
def find_optimal_n_estimators(X, step_points=(25, 50, 100, 150, 200, 300, 400),
                               max_depth=20, min_samples_leaf=20, random_state=RANDOM_STATE):
    Xtr, Xte = X.loc[idx_train], X.loc[idx_test]

    clf = RandomForestClassifier(
        n_estimators=0,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        class_weight="balanced",
        random_state=random_state,
        n_jobs=-1,
        warm_start=True,
    )

    records = []
    t0 = time.time()
    for n in step_points:
        clf.set_params(n_estimators=n)
        clf.fit(Xtr, y_train)  # 이미 만든 나무는 재사용, 늘어난 만큼만 추가 학습
        elapsed = time.time() - t0

        proba_test = clf.predict_proba(Xte)[:, 1]
        auc = roc_auc_score(y_test, proba_test)

        records.append({"n_estimators": n, "AUC": auc, "누적_경과시간(초)": round(elapsed, 1)})
        print(f"n_estimators={n:>4d} | AUC={auc:.4f} | 누적 경과시간={elapsed:.1f}초")

    return pd.DataFrame(records)


progress_df = find_optimal_n_estimators(X_ABC)

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(progress_df["n_estimators"], progress_df["AUC"], marker="o", color="#1E2761")
ax1.set_xlabel("나무 개수 (n_estimators)")
ax1.set_ylabel("Test AUC", color="#1E2761")
ax1.set_title("나무 개수에 따른 AUC 변화 — M4(A+B+C) 기준")

ax2 = ax1.twinx()
ax2.plot(progress_df["n_estimators"], progress_df["누적_경과시간(초)"], marker="s",
          color="#F2A93B", linestyle="--")
ax2.set_ylabel("누적 학습시간(초)", color="#F2A93B")

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "rf_n_estimators_progress.png"), dpi=150)
plt.show()

display(progress_df)

# ↓↓↓ 위 그래프에서 AUC가 평평해지기 시작하는 n_estimators 값을 확인한 뒤
#     아래 N_ESTIMATORS_FINAL 값을 그 지점으로 맞추고 4번부터 이어서 실행
N_ESTIMATORS_FINAL = 200
MAX_DEPTH_FINAL = 20
MIN_SAMPLES_LEAF_FINAL = 20

## 적합 + 평가 함수

In [ ]:
def fit_eval_rf(X, name):
    Xtr, Xte = X.loc[idx_train], X.loc[idx_test]
    clf = RandomForestClassifier(
        n_estimators=N_ESTIMATORS_FINAL,
        max_depth=MAX_DEPTH_FINAL,
        min_samples_leaf=MIN_SAMPLES_LEAF_FINAL,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1,  # 학습 진행상황 실시간 출력
    )
    clf.fit(Xtr, y_train)
    proba_test = clf.predict_proba(Xte)[:, 1]
    auc = roc_auc_score(y_test, proba_test)

    imp = pd.DataFrame({
        "모델": name, "변수": X.columns, "중요도": clf.feature_importances_
    }).sort_values("중요도", ascending=False)
    imp["중요도_비율(%)"] = (imp["중요도"] / imp["중요도"].sum() * 100).round(2)

    print(f"[{name}] AUC={auc:.4f}  변수수={X.shape[1]}")
    print(imp.head(10)[["변수", "중요도", "중요도_비율(%)"]].to_string(index=False))

    return {"name": name, "auc": auc, "proba_test": proba_test, "importance": imp, "model": clf}

## M1~M4 실행

In [ ]:
print("[M1] 역할: 금융정보만으로 얼마나 예측되는지 기준점 마련 (베이스라인)")
m1 = fit_eval_rf(X_A, "M1_A_financial_RF")

# M1 벤치마크 체크: 공개된 유사 분석(EXT_SOURCE 포함 로지스틱)은 대체로 AUC 0.70~0.76
# 우리는 EXT_SOURCE 없이 랜덤포레스트라 범위가 다를 수 있으나, 참고용으로 체크만 출력
BENCHMARK_LOW, BENCHMARK_HIGH = 0.70, 0.76
if not (BENCHMARK_LOW <= m1["auc"] <= BENCHMARK_HIGH):
    print(f"  ※ 참고: M1 AUC({m1['auc']:.4f})가 벤치마크 범위({BENCHMARK_LOW}~{BENCHMARK_HIGH}) 밖입니다. "
          f"단, 이 벤치마크는 EXT_SOURCE 포함·로지스틱 기준이라 랜덤포레스트+EXT_SOURCE 제외 조건에서는 "
          f"직접 비교 기준으로 쓰기 어려움 — 참고만 하고 코드/데이터 자체 이상 여부는 별도로 확인할 것")
else:
    print(f"  ※ M1 AUC({m1['auc']:.4f})가 벤치마크 범위({BENCHMARK_LOW}~{BENCHMARK_HIGH}) 안에 있음")

print("\n[M2] 역할: 비금융정보만의 예측력 참고치 확보 (RQ 직접 검증용 아님)")
m2 = fit_eval_rf(X_B, "M2_B_nonfinancial_RF")
if m2["auc"] >= m1["auc"]:
    print(f"  ※ 참고: M2 AUC({m2['auc']:.4f})가 M1({m1['auc']:.4f})보다 낮지 않음 — 일반적인 예상과 다르므로 확인 필요")
else:
    print(f"  ※ M2 AUC({m2['auc']:.4f}) < M1 AUC({m1['auc']:.4f}) — 예상대로 비금융 단독이 금융보다 낮음")

print("\n[M3] 역할: 금융에 비금융을 더하면 예측력이 오르는지 확인 (RQ1 핵심)")
m3 = fit_eval_rf(X_AB, "M3_A_plus_B_RF")

print("\n[M4] 역할: 금융+새 비금융만으로도 예측력이 오르는지 확인 (RQ2)")
m4 = fit_eval_rf(X_ABC, "M4_A_plus_B_plus_C_RF")

## DeLong test (M3 vs M1, M4 vs M1, M4 vs M3)

In [ ]:
# LightGBM 버전(M1_M3_LightGBM.ipynb)과 동일한 구현 재사용
def _compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T
    return T2


def _fast_delong(preds_sorted_T, m):
    n = preds_sorted_T.shape[1] - m
    pos = preds_sorted_T[:, :m]
    neg = preds_sorted_T[:, m:]
    k = preds_sorted_T.shape[0]
    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m + n])
    for r in range(k):
        tx[r, :] = _compute_midrank(pos[r, :])
        ty[r, :] = _compute_midrank(neg[r, :])
        tz[r, :] = _compute_midrank(preds_sorted_T[r, :])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.cov(v01); sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov


def delong_test(y_true, proba_1, proba_2):
    order = np.argsort(-y_true)
    y_sorted = y_true[order]
    m = int(y_sorted.sum())
    preds = np.vstack([proba_1, proba_2])[:, order]
    aucs, cov = _fast_delong(preds, m)
    auc_diff = aucs[0] - aucs[1]
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    z = auc_diff / np.sqrt(var) if var > 0 else np.nan
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return {"AUC_1": aucs[0], "AUC_2": aucs[1], "AUC_차이": auc_diff, "z": z, "p_value": p}


print("=== DeLong test: M3(A+B) vs M1(A) — RQ1 핵심 검증 ===")
dl_m3_m1 = delong_test(y_test, m3["proba_test"], m1["proba_test"])
print(dl_m3_m1)

print("\n=== DeLong test: M4(A+B+C) vs M3(A+B) — RQ2 핵심 검증 (C 추가 순수 효과) ===")
dl_m4_m3 = delong_test(y_test, m4["proba_test"], m3["proba_test"])
print(dl_m4_m3)

print("\n=== DeLong test: M4(A+B+C) vs M1(A) — 참고용 (비금융 전체 B+C 누적효과, "
      "B·C 효과가 섞여있어 RQ1·RQ2 어느 쪽 직접 검증에도 해당하지 않음) ===")
dl_m4_m1 = delong_test(y_test, m4["proba_test"], m1["proba_test"])
print(dl_m4_m1)

## 유의성 판정 (로드맵 기준 ①~④ 자동 체크)

In [ ]:
# 로드맵 기준: ①AUC 상승 ②통계적 유의(DeLong) ③실질적 상승폭(0.01~0.02p 이상)
#            ④비금융 변수 다수가 p<0.05
# ④는 로지스틱 계수 p-value 기준이라 랜덤포레스트에는 해당 없음 — 대신
# 비금융 변수군의 변수중요도 비중(%)으로 대체 판단 (아래 imp_ratio_new)
EFFECT_SIZE_MIN = 0.01  # 실무적 기준 하한 (0.01~0.02 중 보수적으로 0.01 사용)


def judge_significance(label, auc_base, auc_new, delong_result, imp_ratio_new):
    diff = auc_new - auc_base
    is_up = diff > 0
    is_sig = delong_result["p_value"] < 0.05
    is_practical = diff >= EFFECT_SIZE_MIN

    print(f"\n[{label}] AUC {auc_base:.4f} → {auc_new:.4f} (Δ={diff:+.4f})")
    print(f"  ① AUC 상승          : {'PASS' if is_up else 'FAIL'}")
    print(f"  ② 통계적 유의(DeLong p={delong_result['p_value']:.2e}) : {'PASS' if is_sig else 'FAIL'}")
    print(f"  ③ 실질적 상승폭(≥{EFFECT_SIZE_MIN}) : {'PASS' if is_practical else 'FAIL'}")
    print(f"  ④(대체) 신규 변수군 중요도 비중 : {imp_ratio_new:.2f}% "
          f"(로지스틱 p<0.05 기준 대신 — 랜덤포레스트는 계수가 없어 중요도 비중으로 대체 판단)")

    verdict = "의미있는 개선" if (is_up and is_sig and is_practical) else "기준 미충족 — 재검토 필요"
    print(f"  → 종합 판정: {verdict}")


# M3: B(비금융 기존) 변수군의 중요도 비중
m3_imp = m3["importance"].copy()
m3_imp["군"] = np.where(m3_imp["변수"].isin(A_VARS), "A_금융", "B_기존비금융")
b_ratio_in_m3 = m3_imp.loc[m3_imp["군"] == "B_기존비금융", "중요도_비율(%)"].sum()
judge_significance("M3 vs M1 (RQ1 핵심)", m1["auc"], m3["auc"], dl_m3_m1, b_ratio_in_m3)

# M4: C(신규비금융) 변수군만의 중요도 비중 — RQ2는 C의 순수 기여도를 봐야 하므로
# B(기존비금융)는 제외하고 C만 집계 (④ 대체 지표)
m4_imp_tmp = m4["importance"].copy()
m4_imp_tmp["군"] = m4_imp_tmp["변수"].apply(lambda v: "C_신규비금융" if v in C_VARS else "기타")
c_ratio_in_m4 = m4_imp_tmp.loc[m4_imp_tmp["군"] == "C_신규비금융", "중요도_비율(%)"].sum()

# RQ2 핵심: M3(A+B) 대비 M4(A+B+C) — C를 추가했을 때의 순수 효과
# (M1 vs M4는 B+C를 합친 효과라 C만의 기여를 분리할 수 없어 RQ2 판정에는 부적합 — M3 vs M4가 맞음)
judge_significance("M4 vs M3 (RQ2 핵심 — C 추가 효과)", m3["auc"], m4["auc"], dl_m4_m3, c_ratio_in_m4)

## 최종 요약

In [ ]:
print("=" * 70)
print("최종 요약")
print("=" * 70)

auc_summary = pd.DataFrame([
    {"모델": "M1 (A_금융)", "변수수": len(A_VARS), "AUC": round(m1["auc"], 4)},
    {"모델": "M2 (B_기존비금융)", "변수수": len(B_VARS), "AUC": round(m2["auc"], 4)},
    {"모델": "M3 (A+B)", "변수수": len(A_VARS) + len(B_VARS), "AUC": round(m3["auc"], 4)},
    {"모델": "M4 (A+B+C)", "변수수": len(A_VARS) + len(B_VARS) + len(C_VARS), "AUC": round(m4["auc"], 4)},
])
print(auc_summary.to_string(index=False))

print("\n--- DeLong test ---")
delong_summary = pd.DataFrame([
    {"비교": "M3 vs M1 (RQ1 핵심)", **dl_m3_m1},
    {"비교": "M4 vs M1 (RQ2 보강)", **dl_m4_m1},
    {"비교": "M4 vs M3 (C 추가 효과)", **dl_m4_m3},
])
print(delong_summary.to_string(index=False))

# M4에서 A/B/C 그룹별 변수중요도 합계 — RQ2 보강 근거(B vs C 중요도 비교)
print("\n--- M4 변수중요도의 군별 합계 (B vs C 비교 = RQ2 보강 근거) ---")
m4_imp = m4["importance"].copy()


def tag_group(v):
    if v in A_VARS:
        return "A_금융"
    if v in B_VARS:
        return "B_기존비금융"
    return "C_신규비금융"


m4_imp["군"] = m4_imp["변수"].apply(tag_group)
print(m4_imp.groupby("군")["중요도_비율(%)"].sum().round(2).to_string())

# M3 변수중요도에도 군(A/B) 표시 — M3는 C가 없어서 A_금융/B_기존비금융만 나옴
m3_imp_out = m3["importance"].copy()
m3_imp_out["군"] = m3_imp_out["변수"].apply(tag_group)

# 결과 저장 (변수명만으로는 A/B/C 구분이 안 되므로 "군" 컬럼 포함해서 저장)
auc_summary.to_csv(os.path.join(BASE_DIR, "rf_M1_M4_auc_summary.csv"), index=False)
m3_imp_out.to_csv(os.path.join(BASE_DIR, "rf_M3_feature_importance.csv"), index=False)
m4_imp.to_csv(os.path.join(BASE_DIR, "rf_M4_feature_importance.csv"), index=False)
print(f"\n결과 저장 완료: {BASE_DIR}")